# Algoritmos de optimización - Seminario

**Nombre y Apellidos:** Rodolfo Verde

**Url:** https://github.com/rodolfo-verde/SEMINARIO/tree/main

**Problema:**
> 1. Sesiones de doblaje
> 2. Organizar los horarios de partidos de La Liga
> 3. Combinar cifras y operaciones

**Descripción del problema (enunciado):**

1. *Sesiones de doblaje.* Tenemos 10 actores y 30 tomas que grabar. Cada actor cobra
   una cantidad fija por cada día que tenga que ir al estudio, sin importar cuántas
   tomas haga ese día. Además no se pueden grabar más de 6 tomas en un mismo día. La idea es
   organizar las sesiones para que el gasto en actores sea el menor posible.
2. *Horarios de La Liga.* Hay 20 equipos (3 de categoría A, 11 de B y 6 de C) y 10
   horarios disponibles. Cada partido tiene una audiencia base según las categorías de
   los equipos, luego se multiplica por un coeficiente según el horario y se penaliza
   si coincide con otros partidos. Hay que repartir los partidos de una jornada en los
   horarios para que la audiencia total sea máxima (obligatorio un partido el viernes
   y otro el lunes).
3. *Combinar cifras y operaciones.* Con las cifras del 1 al 9 sin repetir y los signos
   `+ - * /` alternados, ¿cuál es el valor máximo y el mínimo que se puede obtener?
   ¿Y se pueden obtener todos los valores enteros que hay entre los dos?

(*) La respuesta es obligatoria.

In [1]:
import numpy as np
import pandas as pd
from itertools import combinations, permutations
from functools import lru_cache
from math import comb, factorial
import time

# ---------------- Problema 1: doblaje ----------------
matriz = pd.read_csv("data/doblaje_actores_tomas.csv", index_col=0).to_numpy()
n_tomas, n_actores = matriz.shape
print(f"P1 - Matriz de incidencia: {n_tomas} tomas x {n_actores} actores")

# ---------------- Problema 2: La Liga ----------------
cat = pd.read_csv("data/liga_categorias.csv")["categoria"].tolist()
tab = pd.read_csv("data/liga_audiencia_base.csv")
audiencia_base = {(r.categoria_1, r.categoria_2): r.audiencia_millones
                  for r in tab.itertuples()}
coef_horario = dict(zip(pd.read_csv("data/liga_coeficientes_horario.csv")["horario"],
                        pd.read_csv("data/liga_coeficientes_horario.csv")["coeficiente"]))
coinc = dict(zip(pd.read_csv("data/liga_coincidencias.csv")["coincidencias"],
                 pd.read_csv("data/liga_coincidencias.csv")["reduccion_pct"]))
horarios = list(coef_horario)
print(f"P2 - {len(cat)} equipos ({cat.count('A')}A/{cat.count('B')}B/{cat.count('C')}C), "
      f"{len(horarios)} horarios")

# ---------------- Problema 3: cifras ----------------
cifras = list(range(1, 10))
print("P3 - Cifras 1..9 y signos + - * /")

P1 - Matriz de incidencia: 30 tomas x 10 actores
P2 - 20 equipos (3A/11B/6C), 10 horarios
P3 - Cifras 1..9 y signos + - * /


(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




**Respuesta**

**Problema 1 (doblaje).** Sin considerar las restricciones, cada toma se puede
asignar a cualquier día, lo que da 30^30 ≈ 2,06e44 posibilidades (días etiquetados).
Teniendo en cuenta que no puede haber más de 6 tomas por día, el número baja hasta
1,14e37. Lo he calculado con la recurrencia combinatoria de la celda siguiente.

**Problema 2 (La Liga).** Sin restricciones hay 19!! = 654.729.075 emparejamientos
perfectos de los 20 equipos y 10! = 3.628.800 formas de asignarlos a los horarios,
por lo que el total es 19!! x 10! ≈ 2,38e15. Con las restricciones el resultado es
el mismo: al ser 10 partidos y 10 horarios la asignación es una biyección, y como el
viernes y el lunes son obligatorios, se llenan igualmente.

**Problema 3 (cifras).** Sin restricciones habría 9! x 4^8 ≈ 2,38e10 expresiones
(poniendo cualquier signo en cada hueco). Si cada signo se usa exactamente 2 veces,
interpretación que he adoptado, quedan 9! x 2520 ≈ 9,14e8. Si además el resultado
tiene que ser entero, la búsqueda encuentra 756 valores distintos.

In [2]:
# P1: recurrencia combinatoria (asignaciones de n tomas a k dias, 1..6 por dia)
@lru_cache(None)
def W(n, k):
    if n == 0 and k == 0:
        return 1
    if n == 0 or k == 0:
        return 0
    return sum(comb(n, j) * W(n - j, k - 1) for j in range(1, min(6, n) + 1))

print(f"P1 sin restricciones: {n_tomas}^{n_tomas} = {n_tomas ** n_tomas:.3e}")
print(f"P1 con restricciones (<=6/dia): "
      f"{sum(W(n_tomas, k) for k in range(5, n_tomas + 1)):.3e}")

# P2: emparejamientos perfectos de 20 equipos y asignacion a 10 horarios
matchings = factorial(20) // (2 ** 10 * factorial(10))
print(f"P2 sin/con restricciones: {matchings} x 10! = "
      f"{matchings * factorial(10):.3e}")

# P3: formulas combinatorias
print(f"P3 sin restricciones (4^8): {factorial(9) * 4 ** 8:.3e}")
print(f"P3 cada signo 2 veces (2520): "
      f"{factorial(9) * factorial(8) // factorial(2) ** 4:.3e}")

P1 sin restricciones: 30^30 = 2.059e+44
P1 con restricciones (<=6/dia): 1.140e+37
P2 sin/con restricciones: 654729075 x 10! = 2.376e+15
P3 sin restricciones (4^8): 2.378e+10
P3 cada signo 2 veces (2520): 9.145e+08


Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


**Respuesta**

- **P1 (doblaje):** una matriz binaria (numpy, tomas x actores), con un 1 donde el
  actor participa en la toma. Así la consulta de si un actor está en una toma es O(1)
  y las agregaciones se hacen con sum(axis=...). Con 30 x 10 = 300 celdas no hace
  falta una matriz dispersa, esa opción sería más adecuada para miles de tomas.
- **P2 (La Liga):** diccionarios para la audiencia base, el coeficiente del horario y
  la reducción por coincidencias (acceso O(1) por clave), una lista con las
  categorías de los equipos e itertools.combinations para los 190 partidos posibles.
- **P3 (cifras):** un array de numpy de 362.880 x 9 con todas las permutaciones de
  las cifras, de forma que cada secuencia de signos se evalúa sobre todas las
  permutaciones a la vez. Los signos se guardan como tuplas y los valores enteros en
  un set para evitar duplicados.

No ha sido necesario cambiar la elección inicial; con estas estructuras he podido
realizar todos los cálculos.

In [3]:
# P1: matriz binaria
print("P1:", type(matriz).__name__, matriz.shape,
      "| actores de la toma 1:", np.where(matriz[0] == 1)[0] + 1)

# P2: diccionarios
print("P2: audiencia (A,A) =", audiencia_base[("A", "A")],
      "| coef S20 =", coef_horario["S20"])

# P3: permutaciones y secuencias de signos
P = np.array(list(permutations(cifras)), dtype=np.float64)
secuencias = sorted(set(permutations("+-*/" * 2)))
print("P3: matriz de permutaciones", P.shape,
      "| secuencias de signos", len(secuencias))

P1: ndarray (30, 10) | actores de la toma 1: [1 2 3 4 5]
P2: audiencia (A,A) = 2.0 | coef S20 = 1.0


P3: matriz de permutaciones (362880, 9) | secuencias de signos 2520


Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

**Respuesta**

- **P1 (doblaje):** se quiere minimizar el gasto, que es proporcional al número de
  días-actor: min sum_a sum_d z[a,d], donde z[a,d] vale 1 si el actor a cobra el día
  d. Es un problema de minimización.
- **P2 (La Liga):** se maximiza la audiencia total:
  `max sum_p sum_h base(p) * coef(h) * x[p,h]`, con x[p,h] = 1 si el partido p se
  asigna al horario h. Es un problema de maximización.
- **P3 (cifras):** no es exactamente un problema de optimización, sino de
  enumeración: hay que ver qué valores puede tomar la expresión
  v = d1 s1 d2 ... s8 d9 y cuáles son el mínimo y el máximo.

In [4]:
# P1: variables binarias x[t,d] (la toma t se graba el dia d) y
#     z[a,d] (el actor a cobra el dia d); objetivo: minimizar sum(z)
# P2: variables binarias x[p,h] (partido p en horario h);
#     objetivo: maximizar sum(base * coef * x)
# P3: valor de la expresion v(d1, s1, ..., s8, d9);
#     interesan su minimo, su maximo y el conjunto de enteros alcanzables
print("Objetivos definidos en el modelo (ver Respuesta).")

Objetivos definidos en el modelo (ver Respuesta).


Diseña un algoritmo para resolver el problema por fuerza bruta

**Respuesta**

- **P1:** probar todas las particiones de las 30 tomas en días: cada toma se asigna a
  un día ya existente o se crea uno nuevo, y se elige la partición de menor coste.
  Con 30 tomas es inviable (30^30), así que lo he implementado y validado en una
  instancia pequeña de 7 tomas y 4 actores.
- **P2:** probar todos los emparejamientos perfectos de los equipos y todas las
  asignaciones a horarios, calculando la audiencia de cada una. Con 20 equipos es
  inviable (19!! x 10!), así que lo he probado con 6 equipos y 3 horarios.
- **P3:** generar todas las permutaciones de cifras y todas las secuencias de
  signos, construir la expresión y evaluarla con eval(). Con 9 cifras es inviable
  (9,14e8 evaluaciones), así que lo he probado con 5 cifras (2.880 expresiones).

In [5]:
def coste_solucion(M, grupos):
    """Coste de una solucion: suma de actores distintos por dia."""
    return sum(len({a for t in g for a in np.where(M[t] == 1)[0]}) for g in grupos)


def fuerza_bruta_doblaje(M, max_tomas=6):
    """Enumeracion exhaustiva de todas las particiones de tomas en dias."""
    n = len(M)
    mejor_coste, mejor_sol = np.inf, None

    def rec(idx, grupos):
        nonlocal mejor_coste, mejor_sol
        if idx == n:
            c = coste_solucion(M, grupos)
            if c < mejor_coste:
                mejor_coste, mejor_sol = c, [g[:] for g in grupos]
            return
        for g in grupos:
            if len(g) < max_tomas:
                g.append(idx); rec(idx + 1, grupos); g.pop()
        grupos.append([idx]); rec(idx + 1, grupos); grupos.pop()

    rec(0, [])
    return mejor_coste


M_demo = np.array([[1,1,0,0],[1,0,1,0],[0,1,1,0],[1,0,0,1],
                   [0,1,0,1],[1,1,1,0],[0,0,1,1]])
print(f"P1 fuerza bruta (7 tomas, 4 actores): coste optimo = "
      f"{fuerza_bruta_doblaje(M_demo)}")


def fuerza_bruta_liga(cat_sub, hor_sub):
    """Enumeracion exhaustiva para jornadas pequenas (6-8 equipos)."""
    n = len(cat_sub)
    part = list(combinations(range(n), 2))
    mejor = -1.0
    for combo in combinations(part, n // 2):
        if len({e for p in combo for e in p}) != n:
            continue
        for perm in permutations(hor_sub):
            aud = sum(audiencia_base[(min(cat_sub[e1], cat_sub[e2]),
                                      max(cat_sub[e1], cat_sub[e2]))]
                      * coef_horario[h] for (e1, e2), h in zip(combo, perm))
            if aud > mejor:
                mejor = aud
    return mejor


cat6 = ["A"] * 2 + ["B"] * 2 + ["C"] * 2
print(f"P2 fuerza bruta (6 equipos): audiencia optima = "
      f"{fuerza_bruta_liga(cat6, ['S20', 'D18', 'L20']):.2f} M")


def fuerza_bruta_cifras(digs):
    """Busqueda exhaustiva con eval() para un conjunto pequeno de cifras."""
    n = len(digs)
    valores = set()
    for perm in permutations(digs):
        for sec in permutations(["+", "-", "*", "/"], n - 1):
            expr = "".join(f"{d}{s}" for d, s in zip(perm, sec)) + str(perm[-1])
            v = eval(expr)
            if v == int(v):
                valores.add(int(v))
    return valores


vals5 = fuerza_bruta_cifras((1, 2, 3, 4, 5))
print(f"P3 fuerza bruta (5 cifras): {len(vals5)} valores enteros alcanzables")

P1 fuerza bruta (7 tomas, 4 actores): coste optimo = 6
P2 fuerza bruta (6 equipos): audiencia optima = 2.95 M
P3 fuerza bruta (5 cifras): 25 valores enteros alcanzables


Calcula la complejidad del algoritmo por fuerza bruta

**Respuesta**

- **P1:** `O(D^n * n * a)`, con n tomas, D días y a actores (exponencial en n).
- **P2:** `O((n-1)!! * h! * n)`, con n equipos y h horarios (exponencial en n).
- **P3:** `O(n! * m^(n-1) * n)`, con n cifras y m signos (exponencial en n).

Por tanto, para los datos del enunciado la fuerza bruta no es viable en ninguno de
los tres casos, y el crecimiento del tiempo se observa en la celda siguiente.

In [6]:
# Crecimiento empirico del tiempo de cada fuerza bruta
for n_t in (6, 7, 8):
    rng = np.random.default_rng(n_t)
    M = np.zeros((n_t, 4), dtype=int)
    for t in range(n_t):
        M[t, rng.choice(4, rng.integers(1, 4), replace=False)] = 1
    t0 = time.time()
    fuerza_bruta_doblaje(M)
    print(f"P1 fuerza bruta n={n_t}: {time.time() - t0:.2f} s")

for n_eq in (6, 8):
    catn = ["A"] * 2 + ["B"] * 2 + ["C"] * 2
    t0 = time.time()
    fuerza_bruta_liga(catn, ["S20", "D18", "L20"][:n_eq // 2])
    print(f"P2 fuerza bruta n={n_eq} equipos: {time.time() - t0:.2f} s")

for sub in ((1, 2, 3, 4), (1, 2, 3, 4, 5)):
    t0 = time.time()
    fuerza_bruta_cifras(sub)
    print(f"P3 fuerza bruta n={len(sub)} cifras: {time.time() - t0:.3f} s")

P1 fuerza bruta n=6: 0.00 s
P1 fuerza bruta n=7: 0.02 s
P1 fuerza bruta n=8: 0.06 s
P2 fuerza bruta n=6 equipos: 0.00 s
P2 fuerza bruta n=8 equipos: 0.00 s
P3 fuerza bruta n=4 cifras: 0.000 s
P3 fuerza bruta n=5 cifras: 0.016 s


(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

**Respuesta**

He utilizado técnicas distintas para cada problema:

- **P1:** una heurística voraz + búsqueda local (de complejidad polinómica) y un
  modelo MILP exacto con branch-and-bound (HiGHS). Se apoya en un lema sencillo:
  fusionar dos días nunca encarece, porque |A1 U A2| <= |A1| + |A2|. Por tanto, el
  óptimo usa el mínimo número de días, ceil(30/6) = 5, y basta resolver el MILP con
  5 días.
- **P2:** modelo MILP exacto con HiGHS. El branch-and-bound poda con la relajación
  lineal, sin enumerar las 2,38e15 combinaciones (son 1900 variables binarias y se
  resuelve en milisegundos).
- **P3:** evaluación vectorizada con numpy. Separo la expresión en términos `(+/-)` y
  cadenas `(*/)`, respetando la precedencia, y cada secuencia de signos se evalúa
  sobre las 362.880 permutaciones a la vez. El orden de complejidad es el mismo que
  el de la fuerza bruta, pero la constante es mucho menor (de horas a unos 60 s).

In [7]:
from scipy.optimize import milp, LinearConstraint, Bounds


def voraz(M, max_tomas=6):
    """Voraz: cada toma al dia que menos incrementa el coste (o dia nuevo)."""
    n = len(M)
    orden = sorted(range(n), key=lambda t: -M[t].sum())
    dias = []
    for t in orden:
        nuevos = set(np.where(M[t] == 1)[0])
        mejor_inc, mejor_d = None, None
        for d, tomas in enumerate(dias):
            if len(tomas) >= max_tomas:
                continue
            presentes = {a for tt in tomas for a in np.where(M[tt] == 1)[0]}
            inc = len(nuevos - presentes)
            if mejor_inc is None or inc < mejor_inc:
                mejor_inc, mejor_d = inc, d
        if mejor_d is None or len(nuevos) < mejor_inc:
            dias.append([t])
        else:
            dias[mejor_d].append(t)
    return dias


def busqueda_local(M, grupos, max_tomas=6):
    """Mueve cada toma al mejor destino (dia existente o nuevo) hasta converger."""
    grupos = [g[:] for g in grupos if g]
    for _ in range(200):
        mejorado = False
        for i in range(len(grupos)):
            for t in list(grupos[i]):
                if len(grupos[i]) == 1:
                    continue
                grupos[i].remove(t)
                base = coste_solucion(M, grupos)
                mejor, accion = base, None
                for j in range(len(grupos)):
                    if j == i or len(grupos[j]) >= max_tomas:
                        continue
                    grupos[j].append(t)
                    c = coste_solucion(M, grupos)
                    if c < mejor:
                        mejor, accion = c, ("mover", j)
                    grupos[j].remove(t)
                grupos.append([t])
                c = coste_solucion(M, grupos)
                if c < mejor:
                    mejor, accion = c, ("nuevo", None)
                grupos.pop()
                if accion is not None:
                    if accion[0] == "mover":
                        grupos[accion[1]].append(t)
                    else:
                        grupos.append([t])
                    mejorado = True
                else:
                    grupos[i].append(t)
        grupos = [g for g in grupos if g]
        if not mejorado:
            break
    return grupos


def resolver_milp_doblaje(M, n_dias=5, max_tomas=6, time_limit=90):
    """MILP (HiGHS): 5 dias fijos, ruptura de simetria y cotas por actor."""
    n, a = M.shape
    nv_x, nv_z = n * n_dias, a * n_dias
    c = np.zeros(nv_x + nv_z)
    c[nv_x:] = 1.0
    filas, lb, ub = [], [], []

    def add(f, lo, hi):
        filas.append(f); lb.append(lo); ub.append(hi)

    for t in range(n):
        f = np.zeros(nv_x + nv_z); f[t * n_dias:(t + 1) * n_dias] = 1
        add(f, 1, 1)
    for d in range(n_dias):
        f = np.zeros(nv_x + nv_z); f[d::n_dias][:n] = 1
        add(f, -np.inf, max_tomas)
    for a_idx in range(a):
        for d in range(n_dias):
            for t in range(n):
                if M[t, a_idx] == 1:
                    f = np.zeros(nv_x + nv_z)
                    f[t * n_dias + d] = -1
                    f[nv_x + a_idx * n_dias + d] = 1
                    add(f, 0, np.inf)
    for d in range(n_dias - 1):
        f = np.zeros(nv_x + nv_z)
        f[d::n_dias][:n] = 1
        f[(d + 1)::n_dias][:n] = -1
        add(f, 0, np.inf)
    for a_idx in range(a):
        f = np.zeros(nv_x + nv_z)
        f[nv_x + a_idx * n_dias:(nv_x + (a_idx + 1) * n_dias)] = 1
        add(f, int(np.ceil(M[:, a_idx].sum() / max_tomas)), np.inf)

    A = np.array(filas)
    t0 = time.time()
    res = milp(c=c, integrality=np.ones(nv_x + nv_z), bounds=Bounds(0, 1),
               constraints=LinearConstraint(A, lb, ub),
               options={"time_limit": time_limit})
    return res, time.time() - t0


def resolver_milp_liga(categorias, coef, time_limit=60):
    """MILP (HiGHS): maximiza la audiencia de una jornada."""
    n = len(categorias)
    hor = list(coef)
    part = list(combinations(range(n), 2))
    n_p, n_h = len(part), len(hor)
    base = np.array([audiencia_base[(min(categorias[e1], categorias[e2]),
                                     max(categorias[e1], categorias[e2]))]
                     for e1, e2 in part])
    coef_arr = np.array([coef[h] for h in hor])
    c = -(base[:, None] * coef_arr[None, :]).ravel()
    filas, lb, ub = [], [], []

    def add(f, lo, hi):
        filas.append(f); lb.append(lo); ub.append(hi)

    f = np.ones(n_p * n_h); add(f, n // 2, n // 2)
    for h in range(n_h):
        f = np.zeros(n_p * n_h); f[h::n_h] = 1
        add(f, -np.inf, 1)
    for h in (0, n_h - 1):
        f = np.zeros(n_p * n_h); f[h::n_h] = 1
        add(f, 1, np.inf)
    for eq in range(n):
        f = np.zeros(n_p * n_h)
        for p, (e1, e2) in enumerate(part):
            if eq in (e1, e2):
                f[p * n_h:(p + 1) * n_h] = 1
        add(f, -np.inf, 1)

    A = np.array(filas)
    t0 = time.time()
    res = milp(c=c, integrality=np.ones(n_p * n_h), bounds=Bounds(0, 1),
               constraints=LinearConstraint(A, lb, ub),
               options={"time_limit": time_limit})
    return res, time.time() - t0


TOL = 1e-7


def evaluar_secuencia(P, signos):
    """Evalua una secuencia de signos sobre todas las permutaciones (vectorizado)."""
    terminos, signo, idx0, cadena = [], "+", 0, []
    for k, op in enumerate(signos):
        if op in "+-":
            terminos.append((signo, idx0, cadena))
            signo, idx0, cadena = op, k + 1, []
        else:
            cadena.append((k + 1, op))
    terminos.append((signo, idx0, cadena))
    total = np.zeros(P.shape[0])
    for sgn, i0, cadena in terminos:
        v = P[:, i0].copy()
        for idx, op in cadena:
            v = v * P[:, idx] if op == "*" else v / P[:, idx]
        total = total + v if sgn == "+" else total - v
    return total


print("== RESULTADOS CON LOS DATOS DEL ENUNCIADO ==")
t0 = time.time()
sol_v = busqueda_local(matriz, voraz(matriz))
t_heu = time.time() - t0
print(f"P1 voraz + busqueda local : coste = {coste_solucion(matriz, sol_v)} "
      f"({t_heu:.2f} s)")

res1, t1 = resolver_milp_doblaje(matriz)
print(f"P1 MILP exacto            : coste = {res1.fun:.0f}, "
      f"optimo demostrado = {res1.success} (gap {res1.mip_gap:.4f}, {t1:.0f} s)")

res2, t2 = resolver_milp_liga(cat, coef_horario)
print(f"P2 MILP exacto            : audiencia = {-res2.fun:.2f} M ({t2:.2f} s)")

valores_enteros = set()
val_min, val_max = np.inf, -np.inf
t0 = time.time()
for sec in secuencias:
    v = evaluar_secuencia(P, sec)
    if v.min() < val_min:
        val_min = v.min()
    if v.max() > val_max:
        val_max = v.max()
    r = np.round(v)
    valores_enteros.update(r[np.abs(v - r) < TOL].astype(np.int64).tolist())
t3 = time.time() - t0
print(f"P3 busqueda completa      : min = {val_min}, max = {val_max}, "
      f"enteros = {len(valores_enteros)} ({t3:.0f} s)")

== RESULTADOS CON LOS DATOS DEL ENUNCIADO ==
P1 voraz + busqueda local : coste = 35 (0.00 s)


P1 MILP exacto            : coste = 27, optimo demostrado = True (gap 0.0000, 44 s)
P2 MILP exacto            : audiencia = 7.22 M (0.06 s)


P3 busqueda completa      : min = -507.9, max = 514.1666666666667, enteros = 756 (59 s)


(*)Calcula la complejidad del algoritmo

**Respuesta**

- **P1:** el voraz + búsqueda local es `O(n^2 * a)` en tiempo (polinómico) y O(n) en
  memoria. El MILP en el peor caso es exponencial, pero con la relajación lineal y
  las cotas resuelve esta instancia (200 variables binarias, 5 días) en ~30 s.
- **P2:** el MILP tiene C(20,2) x 10 = 1900 variables binarias y 32 restricciones, y
  se resuelve en milisegundos.
- **P3:** `O(N * m)` en tiempo (N = 362.880 permutaciones, m = 2520 secuencias) y O(N)
  en memoria con la evaluación vectorizada; unos 60 s frente a las horas de eval().

Los tiempos medidos se muestran en la celda siguiente.

In [8]:
print("Tiempos medidos con los datos del enunciado:")
print(f"  P1 voraz + busqueda local : {t_heu:.2f} s")
print(f"  P1 MILP exacto            : {t1:.0f} s")
print(f"  P2 MILP exacto            : {t2:.2f} s")
print(f"  P3 busqueda vectorizada   : {t3:.0f} s")

print("\nComplejidades asintoticas:")
print("  P1 voraz+LS: O(n^2 * a)      |  P1 MILP: exponencial en el peor caso")
print("  P2 MILP: 1900 variables, milisegundos")
print("  P3: O(N * m) tiempo, O(N) memoria  (N = 362880, m = 2520)")

Tiempos medidos con los datos del enunciado:
  P1 voraz + busqueda local : 0.00 s
  P1 MILP exacto            : 44 s
  P2 MILP exacto            : 0.06 s
  P3 busqueda vectorizada   : 59 s

Complejidades asintoticas:
  P1 voraz+LS: O(n^2 * a)      |  P1 MILP: exponencial en el peor caso
  P2 MILP: 1900 variables, milisegundos
  P3: O(N * m) tiempo, O(N) memoria  (N = 362880, m = 2520)


Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

**Respuesta**

- **P1:** una matriz sintética de 30 tomas x 10 actores con 2 a 5 actores por toma
  (mismas dimensiones, otra estructura).
- **P2:** dos escenarios: 20 equipos con 2A/10B/8C y una liga pequeña de 12 equipos
  (2A/4B/6C).
- **P3:** un subconjunto aleatorio de 6 cifras del 1 al 9.

In [9]:
rng = np.random.default_rng(7)


def generar_matriz_aleatoria(seed=7):
    rng2 = np.random.default_rng(seed)
    M = np.zeros((30, 10), dtype=int)
    for t in range(30):
        k = int(rng2.integers(2, 6))
        M[t, rng2.choice(10, k, replace=False)] = 1
    return M


M_rand = generar_matriz_aleatoria()
cat_alt = ["A"] * 2 + ["B"] * 10 + ["C"] * 8
cat_12 = ["A"] * 2 + ["B"] * 4 + ["C"] * 6
conjunto6 = tuple(sorted(int(x) for x in rng.choice(range(1, 10), 6, replace=False)))

print("P1 datos aleatorios:", M_rand.shape,
      "| tomas con", M_rand.sum(axis=1).min(), "a", M_rand.sum(axis=1).max(),
      "actores")
print("P2 escenarios: 20 equipos 2A/10B/8C y 12 equipos 2A/4B/6C")
print("P3 cifras aleatorias:", conjunto6)

P1 datos aleatorios: (30, 10) | tomas con 2 a 5 actores
P2 escenarios: 20 equipos 2A/10B/8C y 12 equipos 2A/4B/6C
P3 cifras aleatorias: (4, 5, 6, 7, 8, 9)


Aplica el algoritmo al juego de datos generado

**Respuesta**

He utilizado los datos aleatorios del apartado anterior y les he aplicado los mismos
algoritmos: el MILP al problema de doblaje y a la liga, y la búsqueda vectorizada a
las cifras. Los resultados se muestran en la celda siguiente.

In [10]:
print("== RESULTADOS CON LOS DATOS ALEATORIOS ==")
sol_r = busqueda_local(M_rand, voraz(M_rand))
print(f"P1 voraz + busqueda local : coste = {coste_solucion(M_rand, sol_r)}")

res1r, t1r = resolver_milp_doblaje(M_rand, time_limit=60)
print(f"P1 MILP exacto            : coste = {res1r.fun:.0f}, "
      f"optimo = {res1r.success} (gap {res1r.mip_gap:.3f}, {t1r:.0f} s)")

for nombre, c in [("2A/10B/8C ", cat_alt), ("12 equipos", cat_12)]:
    res2r, t2r = resolver_milp_liga(c, coef_horario)
    print(f"P2 MILP {nombre}          : audiencia = {-res2r.fun:.2f} M ({t2r:.2f} s)")

P6 = np.array(list(permutations(conjunto6)), dtype=np.float64)
sec6 = sorted(set(permutations("+-*/" * 5, 5)))
vals6 = set()
for sec in sec6:
    v = evaluar_secuencia(P6, sec)
    r = np.round(v)
    vals6.update(r[np.abs(v - r) < TOL].astype(np.int64).tolist())
print(f"P3 cifras {conjunto6}     : {len(vals6)} enteros, "
      f"rango [{min(vals6)}, {max(vals6)}]")

== RESULTADOS CON LOS DATOS ALEATORIOS ==
P1 voraz + busqueda local : coste = 43


P1 MILP exacto            : coste = 33, optimo = False (gap 0.303, 60 s)


P2 MILP 2A/10B/8C           : audiencia = 6.60 M (0.40 s)
P2 MILP 12 equipos          : audiencia = 4.51 M (0.03 s)
P3 cifras (4, 5, 6, 7, 8, 9)     : 882 enteros, rango [-15116, 60480]


Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

**Respuesta**

- Documentación de scipy.optimize.milp (solver HiGHS):
  https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html
- Página de HiGHS: https://highs.dev/
- Documentación de Python (itertools, eval): https://docs.python.org/3/
- Documentación de numpy: https://numpy.org/doc/stable/
- Apuntes de la asignatura Algoritmos de Optimización (03MIAR), VIU.
- Enunciado del trabajo práctico (VIU-03MIAR-Trabajo_Práctico.pdf).

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

**Respuesta**

- **P1 (doblaje):** se podría complicar con costes distintos por actor, límites de
  horas o disponibilidades. Con cientos de tomas el MILP crece mucho y convendría
  generación de columnas o metaheurísticas (recocido simulado, búsqueda tabú). Las
  evaluaciones se pueden paralelizar.
- **P2 (La Liga):** se podría considerar audiencia por equipos concretos, limitaciones
  de TV o ventanas por equipo. Programar la liga completa (38 jornadas con coherencia
  casa/fuera) es bastante más complejo: se usaría descomposición, metaheurísticas o
  constraint programming (OR-Tools CP-SAT). Si se permitieran varias coincidencias, la
  corrección habría que linealizarla con variables auxiliares.
- **P3 (cifras):** si se permite el cero, los paréntesis o repetir cifras cambia el
  espacio de búsqueda. Con paréntesis se podría usar programación dinámica sobre
  subexpresiones. Al crecer n el espacio n! crece muy rápido: se puede podar por
  bloques o paralelizar por secuencia de signos. Usar aritmética racional exacta
  eliminaría las tolerancias.